# PageZero SRE Agent — GRPO Training

Trains a Qwen3-0.6B agent to diagnose and fix production incidents using GRPO.

**Stack:** TRL 1.2+ | transformers 5.x | bitsandbytes 4-bit | openenv-core

**Run all cells in order from top to bottom.**

## 1) Install Dependencies

In [1]:
import subprocess
import sys
from importlib import metadata

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'pip'], check=True)

# Core packages — no unsloth, no vllm (they conflict with TRL 1.x / transformers 5.x)
core_packages = [
    'trl>=0.29.0',
    'transformers>=5.2.0',
    'huggingface-hub>=1.5.0,<2.0',
    'datasets',
    'accelerate',
    'bitsandbytes',
    'matplotlib',
    'pandas',
    'peft',
    'jmespath',
    'nest_asyncio',
    'liger-kernel',
    'trackio',
    'openenv-core[core]>=0.2.1',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', *core_packages], check=True)

# Verify
print('-' * 40)
for pkg in ['transformers', 'trl', 'huggingface-hub', 'datasets', 'peft', 'bitsandbytes', 'liger-kernel', 'trackio']:
    try:
        print(f'{pkg}: {metadata.version(pkg)}')
    except Exception:
        print(f'{pkg}: NOT INSTALLED')

----------------------------------------
transformers: 5.6.2
trl: 1.2.0
huggingface-hub: 1.12.0
datasets: 4.8.4
peft: 0.19.1
bitsandbytes: 0.49.2
liger-kernel: 0.7.0
trackio: 0.25.1





### ⚠️ After running the cell above: Runtime → Restart runtime, then continue from Cell 2

## 2) Configuration

In [2]:
import os

IN_COLAB = 'COLAB_GPU' in os.environ

# --- HuggingFace token (optional unless pushing to hub) ---
try:
    from google.colab import userdata
    if 'HF_TOKEN' not in os.environ:
        token = userdata.get('HF_TOKEN')
        if token:
            os.environ['HF_TOKEN'] = token
except Exception:
    pass

if 'HF_TOKEN' not in os.environ:
    print('WARNING: HF_TOKEN not found. Needed only for push_to_hub.')

# --- Backend mode selector ---
BACKEND_MODE = 'deployed_space'  # local_same_machine | local_via_tunnel | deployed_space | custom_url

LOCAL_SAME_MACHINE_URL = 'http://localhost:8000'
LOCAL_TUNNEL_URL = 'https://brave-pots-throw.loca.lt/'
DEPLOYED_SPACE_URL = 'https://neokazuha-pagezero-agent.hf.space'
CUSTOM_ENV_URL = os.environ.get('PAGEZERO_ENV_URL', '').strip()

if BACKEND_MODE == 'local_same_machine':
    ENV_URL = LOCAL_SAME_MACHINE_URL
elif BACKEND_MODE == 'local_via_tunnel':
    ENV_URL = LOCAL_TUNNEL_URL
elif BACKEND_MODE == 'deployed_space':
    ENV_URL = DEPLOYED_SPACE_URL
elif BACKEND_MODE == 'custom_url':
    if not CUSTOM_ENV_URL:
        raise ValueError("BACKEND_MODE='custom_url' but PAGEZERO_ENV_URL is empty.")
    ENV_URL = CUSTOM_ENV_URL
else:
    raise ValueError(f'Unknown BACKEND_MODE: {BACKEND_MODE}')

if IN_COLAB and BACKEND_MODE == 'local_same_machine':
    print('WARNING: localhost points to the Colab VM, not your laptop.')
    print('Use local_via_tunnel or deployed_space for Colab.')

# --- Repo / checkout config ---
REPO_URL = 'https://huggingface.co/spaces/neokazuha/pagezero_agent'
USE_EXISTING_CHECKOUT = not IN_COLAB
EXISTING_CHECKOUT_DIR = os.getcwd() if not IN_COLAB else '/content/PageZero'
CLONE_DIR = '/content/PageZero' if IN_COLAB else EXISTING_CHECKOUT_DIR

# --- Model config ---
USE_UNSLOTH_MODEL = False  # True will try 4-bit Unsloth model id
UNSLOTH_MODEL_ID = 'unsloth/Qwen3-0.6B-bnb-4bit'
BASE_MODEL_ID = 'Qwen/Qwen3-0.6B'
MODEL_ID = UNSLOTH_MODEL_ID if USE_UNSLOTH_MODEL else BASE_MODEL_ID
FALLBACK_MODEL_ID = BASE_MODEL_ID

# --- Training hyperparameters (kube-sre-gym memory-and-speed recipe) -----
# Per-stage budget — total dataset size for the trainer.train() call. Mastery
# gates may end the stage early once the agent's recent_success_rate clears
# the tier threshold, so this is an *upper bound*, not a fixed count.
NUM_EPISODES = 4
# GRPO group g — sampled completions per prompt that compete inside a group.
# Larger g → lower advantage variance, but multiplies GPU memory and rollout
# wallclock. T4-safe = 4-6; we default to 4 to keep early iterations cheap.
NUM_GENERATIONS = 4
G_PARAM = NUM_GENERATIONS
# Tier 1 (rollout shape): allow enough tokens per *accumulated* completion so
# tool results + next <tool_call> fit; 256 caused almost all episodes to stop
# after 1 env step. Raise MAX_TOTAL_TOKENS in step with max_completion_length.
MAX_TURNS = 8
MAX_NEW_TOKENS = 2048
# Sampling — lower temperature reduces mode-collapse on diagnose_root_cause.
TEMPERATURE = 0.6
TOP_P = 0.9  # passed to GRPOConfig when the installed trl version supports top_p
# Hard cap on accumulated rollout tokens to prevent backward-pass OOM.
MAX_TOTAL_TOKENS = 12288
VLLM_MODE = 'colocate'
OUTPUT_DIR_BASE = 'outputs/pagezero-colab'

# --- KL-divergence regularization (β) ---
# Static β=0.01 (kube-sre-gym proved adaptive scheduling is unnecessary once
# the reward signal has variance). The old 0.04 over-damped exploration.
KL_BETA = 0.01

# --- Mastery-gated curriculum stages -------------------------------------
# Each stage defines a task pool. The MasteryCurriculum advances stage→stage
# only when ``recent_success_rate >= advance_rate`` over ``min_episodes``
# (kube-sre-gym pattern). ``episodes`` here is an upper-bound budget per
# stage; the gate may close the stage early on fast-track.
CURRICULUM_STAGES = [
    {
        'name': 'easy',
        'task_ids': ['task_1', 'task_2', 'task_3', 'task_4', 'task_5'],
        'episodes': NUM_EPISODES,
        'min_episodes': 5,
        'advance_rate': 0.6,
    },
    {
        'name': 'medium',
        'task_ids': ['task_6', 'task_7', 'task_8', 'task_9', 'task_10'],
        'episodes': NUM_EPISODES,
        'min_episodes': 8,
        'advance_rate': 0.65,
    },
    {
        'name': 'hard',
        'task_ids': ['task_11', 'task_12'],
        'episodes': NUM_EPISODES,
        'min_episodes': 10,
        'advance_rate': 0.7,
    },
]

# --- Trajectory + top-level reward log ----------------------------------
# Single canonical CSV at the output dir root (no per-stage subdir burying)
# plus a JSONL with the full trajectory per episode.
REWARD_LOG_NAME = 'reward_log.csv'
TRAJECTORY_LOG_NAME = 'trajectories.jsonl'

# Penalty applied when a completion parses to zero tool calls (the env was
# never stepped). Mirrors kube-sre-gym's -0.5 for invalid commands.
NO_TOOL_PENALTY = -0.5

# --- Reward audit guards ---
# Sized for kube-sre-gym-style high-variance terminals: -2.0 wipe on timeout,
# up to ~+8 on confirmed resolution. We give ~25% headroom on the upper
# bound so genuine extremes are never clipped.
REWARD_AUDIT_MIN = -3.0
REWARD_AUDIT_MAX = 10.0

# --- Hub push config ---
PUSH_TO_HUB = False
HUB_REPO = 'neokazuha/pagezero_agent'

print(f'IN_COLAB        : {IN_COLAB}')
print(f'BACKEND_MODE    : {BACKEND_MODE}')
print(f'ENV_URL         : {ENV_URL}')
print(f'MODEL_ID        : {MODEL_ID}')
print(f'EPISODES/STAGE  : {NUM_EPISODES} (upper bound; mastery gate may end early)')
print(f'GENERATIONS (g) : {NUM_GENERATIONS}')
print(f'MAX_TURNS       : {MAX_TURNS}')
print(f'MAX_NEW_TOKENS  : {MAX_NEW_TOKENS}')
print(f'TEMPERATURE     : {TEMPERATURE}')
print(f'TOP_P           : {TOP_P}')
print(f'KL β            : {KL_BETA}')
print(f'CURRICULUM      : {[s["name"] for s in CURRICULUM_STAGES]}')

IN_COLAB        : True
BACKEND_MODE    : deployed_space
ENV_URL         : https://neokazuha-pagezero-agent.hf.space
MODEL_ID        : Qwen/Qwen3-0.6B
EPISODES/STAGE  : 4 (upper bound; mastery gate may end early)
GENERATIONS (g) : 4
MAX_TURNS       : 8
MAX_NEW_TOKENS  : 512
TEMPERATURE     : 0.6
TOP_P           : 0.9
KL β            : 0.01
CURRICULUM      : ['easy', 'medium', 'hard']


## 3) Clone Repo & Install

In [3]:
import os
import subprocess

if IN_COLAB:
    if not os.path.exists(CLONE_DIR):
        subprocess.run(['git', 'clone', REPO_URL, CLONE_DIR], check=True)
    os.chdir(CLONE_DIR)
    subprocess.run(['pip', 'install', '-q', '-e', '.[train]'], check=True)

print(f'Working directory: {os.getcwd()}')

Working directory: /content/PageZero


## 4) Test Backend Connection

In [4]:
import requests
import asyncio
import sys, os

if os.path.abspath('..') not in sys.path:
    sys.path.append(os.path.abspath('..'))

from PageZero.client import PageZeroEnvClient
from PageZero.models import PageZeroAction, PageZeroObservation
from pydantic import ConfigDict

# Temporarily modify PageZeroObservation to allow extra fields
# This addresses the ValidationError caused by unexpected fields from the backend.
PageZeroObservation.model_config = ConfigDict(extra='allow')

print(f'Testing backend: {ENV_URL}')
url = ENV_URL.rstrip('/') + '/health'
try:
    r = requests.get(url, timeout=10)
    print(f'{url} -> {r.status_code}')
except Exception as e:
    print(f'{url} -> FAILED: {e}')

async def run_env_interaction():
    env = PageZeroEnvClient(base_url=ENV_URL, message_timeout_s=300.0)
    try:
        reset_result = await env.reset()
        obs = reset_result.observation
        print('Reset successful')
        print(f'Step {obs.step}/{obs.max_steps}')
        print(f'Active alerts: {obs.active_alerts}')

        step_result = await env.step(PageZeroAction(tool='check_alerts', args={}))
        print(f'Step reward: {step_result.reward}')
        print(f'Done: {step_result.done}')
    finally:
        await env.close()

await run_env_interaction()


Testing backend: https://neokazuha-pagezero-agent.hf.space
https://neokazuha-pagezero-agent.hf.space/health -> 200
Reset successful
Step 0/15
Active alerts: ['CRITICAL: Redis memory usage > 95%, OOM errors in app logs — check redis_info and flush orphaned keys']
Step reward: 0.2
Done: False


## 5) Import Training Utilities

In [5]:
import csv
import logging
import os
import sys
from datetime import datetime
from pathlib import Path

import nest_asyncio
nest_asyncio.apply()

from datasets import Dataset
from transformers import AutoTokenizer
from trl import GRPOConfig, GRPOTrainer
from peft import LoraConfig

if os.path.abspath('..') not in sys.path:
    sys.path.append(os.path.abspath('..'))

from PageZero.train import SYSTEM_PROMPT, plot_rewards, build_grpo_dataset

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger(__name__)

print('Imports OK.')
print('System prompt preview:')
print(SYSTEM_PROMPT[:220] + '...')

Imports OK.
System prompt preview:
You are a Staff SRE on-call. Diagnose and fix the cascading incident across the Application, PostgreSQL, and Redis cache.

To use a tool, you MUST use this exact format:
<tool_call>
{"name": "check_alerts", "arguments": ...


## 6) Define PageZeroToolEnv (async-safe for Colab)

In [6]:
import asyncio, threading, time, os, json
from typing import Any, Dict

from PageZero.client import PageZeroEnvClient
from PageZero.models import PageZeroAction
# Canonical wrapper + curriculum + logger live in PageZero.train so the
# notebook and CLI never drift. We subclass below only to swap the
# synchronous WebSocket client for a Colab-safe asyncio bridge.
from PageZero.train import (
    PageZeroToolEnv as _CanonicalToolEnv,
    MasteryCurriculum,
    RewardLogger,
    make_reward_total,
    reward_diagnosis_metric,
    reward_fix_metric,
)

# ── Background event loop (avoids Jupyter's "loop already running" error) ──
_bg_loop = asyncio.new_event_loop()
threading.Thread(target=_bg_loop.run_forever, daemon=True).start()


def _sync(coro, timeout=360):
    """Submit async coroutine to background loop and block until done."""
    return asyncio.run_coroutine_threadsafe(coro, _bg_loop).result(timeout=timeout)


class PageZeroToolEnv(_CanonicalToolEnv):
    """Colab-safe variant of the canonical wrapper.

    Overrides only ``__init__``, ``reset``, ``_run_tool`` and ``_close_all``
    to bridge the async openenv WebSocket client into the synchronous
    interface that TRL / GRPOTrainer expects. ALL reward/trajectory/
    feedback/curriculum logic comes from the parent class so the notebook
    and CLI behave identically.

    Curriculum hooks live on the parent as ``_set_stage`` / ``_close_all`` so TRL
    does not treat them as agent-callable tools.
    """

    BASE_URL: str = ENV_URL
    VERBOSE_REWARDS: bool = bool(int(os.environ.get('PZ_VERBOSE_REWARDS', '1')))

    def __init__(self) -> None:
        """Create a Colab-safe env wrapper without opening the sync WebSocket client."""
        # Bypass parent __init__ — it creates a sync client we don't want.
        self._client = None
        self.total_reward = 0.0
        self.diagnosis_reward = 0.0
        self.fix_reward = 0.0
        self.terminal_reward = 0.0
        self.is_done = False
        self._episode_logged = False
        self.trajectory = []
        self.task_id = ''
        self.scenario_name = ''
        self.last_sla_status = 'OK'
        self.is_resolved = False
        self.stack_healthy = None
        self.start_ts = time.time()
        self.end_ts = None
        self._last_feedback = ''
        self._last_command = ''
        self._last_output_snippet = ''
        self._last_reward = 0.0
        type(self)._instances.append(self)

    @classmethod
    def _close_all(cls) -> None:
        """Close every async WebSocket client tracked by this environment subclass.

        Returns:
            None: This is a class-level cleanup side effect only.
        """
        for env in cls._instances:
            try:
                if env._client is not None:
                    _sync(env._client.close(), timeout=10)
            except Exception:
                pass
        cls._instances.clear()

    def reset(self, **kwargs: Any) -> str | None:
        """Start a new episode and return the initial observation text for the model.

        Args:
            kwargs: Optional keyword arguments for future server reset hooks (unused here).

        Returns:
            Formatted observation string appended to the user prompt, or None if the integration passes no observation (rare).
        """
        # Tear down any prior socket and start a fresh one per episode.
        if self._client is not None:
            try:
                _sync(self._client.close(), timeout=10)
            except Exception:
                pass
            self._client = None
            time.sleep(0.3)
        self._client = PageZeroEnvClient(base_url=self.BASE_URL, message_timeout_s=300.0)

        # Pick task_id from the curriculum (preferred) or stage pool fallback.
        chosen_task_id = ''
        if self.CURRICULUM is not None:
            chosen_task_id = self.CURRICULUM.pick_task_id(self.STAGE)
        elif self.STAGE_TASK_IDS:
            import random
            chosen_task_id = random.choice(self.STAGE_TASK_IDS)

        try:
            result = _sync(self._client.reset(task_id=chosen_task_id)) if chosen_task_id \
                     else _sync(self._client.reset())
        except TypeError:
            result = _sync(self._client.reset())
            chosen_task_id = ''

        self.total_reward = 0.0
        self.diagnosis_reward = 0.0
        self.fix_reward = 0.0
        self.terminal_reward = 0.0
        self.is_done = bool(result.done)
        self._episode_logged = False
        self.trajectory = []
        self.task_id = chosen_task_id
        self.scenario_name = (getattr(result.observation, 'tool_output', '') or '')[:120]
        self.last_sla_status = getattr(result.observation, 'sla_status', 'OK') or 'OK'
        self.is_resolved = False
        self.stack_healthy = None
        self.start_ts = time.time()
        self.end_ts = None
        self._last_feedback = ''
        self._last_command = ''
        self._last_output_snippet = ''
        self._last_reward = 0.0

        if self.VERBOSE_REWARDS:
            print(f'[env-reset] stage={self.STAGE} task={chosen_task_id or "auto"} '
                  f'tier={self.CURRICULUM.get_tier_name() if self.CURRICULUM else "n/a"}')
        return self._format_observation(result.observation, reward=0.0)

    def _run_tool(self, tool: str, args: Dict[str, Any]) -> str:
        """Execute one tool call via the async WebSocket client (internal bridge)."""
        if self.is_done and tool != 'done':
            raise ValueError('Episode already done.')

        # One reconnect retry on transient WebSocket drops.
        for attempt in range(2):
            try:
                result = _sync(self._client.step(PageZeroAction(tool=tool, args=args)))
                break
            except Exception as e:
                if attempt == 0 and 'Closed' in type(e).__name__:
                    print('  [reconnect] WebSocket dropped, reconnecting...')
                    self._client = PageZeroEnvClient(base_url=self.BASE_URL,
                                                    message_timeout_s=300.0)
                    _sync(self._client.reset())
                    continue
                raise

        reward = float(result.reward or 0.0)
        was_done = bool(result.done)
        obs = result.observation

        canonical_final = getattr(obs, 'final_score', None)
        try:
            canonical_final = float(canonical_final) if canonical_final is not None else None
        except Exception:
            canonical_final = None

        per_step, terminal = self._split_terminal(reward, was_done, canonical_final)
        if was_done:
            self.terminal_reward = terminal

        self.total_reward += reward
        self.is_done = was_done

        # Trust the env's own is_fix_step flag (set by the auto-detect-fix path).
        is_fix = bool(getattr(obs, 'is_fix_step', False)) or tool in (
            'pg_cancel_query', 'pg_create_index', 'pg_vacuum',
            'redis_flush_db', 'docker_restart', 'rollback_deploy',
        )
        if is_fix:
            self.fix_reward += per_step
        else:
            self.diagnosis_reward += per_step

        sla_status = getattr(obs, 'sla_status', 'OK') or 'OK'
        self.last_sla_status = sla_status
        snippet = (getattr(obs, 'tool_output', '') or '')[:240]

        feedback = getattr(obs, 'judge_feedback', None) or getattr(obs, 'hint', '') or ''
        self._last_command = json.dumps({'name': tool, 'arguments': args}, default=str)[:200]
        self._last_feedback = str(feedback or '')
        self._last_output_snippet = snippet
        self._last_reward = reward

        self.trajectory.append({
            'step': len(self.trajectory) + 1,
            'tool': tool,
            'args': args,
            'reward': reward,
            'per_step_reward': per_step,
            'terminal_reward_component': terminal if was_done else 0.0,
            'is_fix': is_fix,
            'phase': getattr(obs, 'phase', None),
            'repeat_count': getattr(obs, 'repeat_count', 1),
            'sla_status': sla_status,
            'is_done': was_done,
            'stack_healthy': getattr(obs, 'stack_healthy', None),
            'judge_feedback': str(feedback or '')[:300],
            'output_snippet': snippet,
        })

        if was_done:
            self.end_ts = time.time()
            self.stack_healthy = getattr(obs, 'stack_healthy', None)
            self.is_resolved = bool(self.stack_healthy)

        if self.VERBOSE_REWARDS:
            tag = 'FIX' if is_fix else 'DIAG'
            term_note = f' term={terminal:+.3f}' if was_done else ''
            print(
                f'  [step {len(self.trajectory):>2}] {tag:<4s} {tool:<22s} '
                f'r={reward:+.3f} cum={self.total_reward:+.3f}{term_note}'
            )

        return self._format_observation(obs, reward=reward)


# Wire ENV_URL into the canonical wrapper class attribute.
PageZeroToolEnv.BASE_URL = ENV_URL
PageZeroToolEnv.NO_TOOL_PENALTY = NO_TOOL_PENALTY


# ── Quick smoke test ──────────────────────────────────────────────────────
print('Smoke-testing PageZeroToolEnv...')
_test_env = PageZeroToolEnv()
_test_obs = _test_env.reset()
print(f'Reset OK. Observation preview:\n{_test_obs[:300]}...')
_sync(_test_env._client.close(), timeout=10)
PageZeroToolEnv._instances.clear()
print('Smoke test PASSED ✓')


Smoke-testing PageZeroToolEnv...
[env-reset] stage=default task=auto tier=n/a
Reset OK. Observation preview:
INCIDENT CONTEXT:
  task_id: auto
  scenario: 🚨 ALERT: CRITICAL: Redis memory usage > 95%, OOM errors in app logs — check redis_info and flush orphaned keys

TOOL OUTPUT:
🚨 ALERT: CRITICAL: Redis memory usage > 95%, OOM errors in app logs — check redis_info and flush orphaned keys

CURRENT ALERTS:
C...
Smoke test PASSED ✓


## 7) Build GRPO Trainer

In [19]:
import torch, gc, csv, json, math
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from trl import GRPOConfig, GRPOTrainer
from peft import LoraConfig
from datasets import Dataset
from transformers import AutoTokenizer
from pathlib import Path
from datetime import datetime
import trackio

# --- 1. PREVENT OOM on re-runs ---
try:
    if 'trainer' in locals():
        del trainer
    if 'model' in locals():
        del model
    gc.collect()
    torch.cuda.empty_cache()
except Exception:
    pass

# Help PyTorch reuse fragmented GPU memory (kube-sre-gym recipe).
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

# --- 2. Tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# --- 3. Output dir + canonical top-level reward logger -------------------
timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
output_dir = Path(f'{OUTPUT_DIR_BASE}-{timestamp}')
output_dir.mkdir(parents=True, exist_ok=True)

# Single top-level reward_log.csv + trajectories.jsonl (no per-stage burying).
reward_logger = RewardLogger(output_dir, csv_name=REWARD_LOG_NAME,
                             jsonl_name=TRAJECTORY_LOG_NAME)

# Mastery-gated curriculum shared across stages.
stage_pools = {s['name']: list(s['task_ids']) for s in CURRICULUM_STAGES}
mastery_curriculum = MasteryCurriculum(stage_task_pools=stage_pools)

# Wire the wrapper to the logger + curriculum (class-level so every new
# environment_factory() call picks them up).
PageZeroToolEnv.REWARD_LOGGER = reward_logger
PageZeroToolEnv.CURRICULUM = mastery_curriculum

print(f'Output dir       : {output_dir}')
print(f'Reward log       : {reward_logger.csv_path}')
print(f'Trajectory log   : {reward_logger.jsonl_path}')
print(f'Curriculum tier  : {mastery_curriculum.get_tier_name()}')

# --- 4. Reward funcs (single primary; diag/fix kept as logging metrics) ---
# TRL sums all reward_funcs by default — passing diagnosis/fix/total caused
# double-counting in earlier runs. We pass *only* reward_total. The other
# two are still computed by the wrapper and surfaced in the CSV.
reward_total = make_reward_total(no_tool_penalty=NO_TOOL_PENALTY)

def _audit(values, label):
    """Clip / sanitize a reward batch and emit per-call stats."""
    cleaned = []
    for v in values:
        try:
            v = float(v)
        except Exception:
            v = 0.0
        if math.isnan(v) or math.isinf(v):
            print(f'  [reward-audit:{label}] WARN non-finite reward → 0.0')
            v = 0.0
        if v < REWARD_AUDIT_MIN or v > REWARD_AUDIT_MAX:
            print(f'  [reward-audit:{label}] WARN clip {v:+.3f} → '
                  f'[{REWARD_AUDIT_MIN}, {REWARD_AUDIT_MAX}]')
            v = max(REWARD_AUDIT_MIN, min(REWARD_AUDIT_MAX, v))
        cleaned.append(v)
    if cleaned:
        mu = sum(cleaned) / len(cleaned)
        var = sum((x - mu) ** 2 for x in cleaned) / max(1, len(cleaned) - 1)
        sd = var ** 0.5
        print(f'  [reward-audit:{label}] n={len(cleaned)} mean={mu:+.3f} '
              f'std={sd:.3f} min={min(cleaned):+.3f} max={max(cleaned):+.3f}')
    return cleaned

def reward_total_audited(completions=None, environments=None, **kwargs):
    return _audit(reward_total(completions=completions,
                               environments=environments, **kwargs),
                  'total')

# --- 5. Dataset (per-row EPISODE_BRIEF + episode_task_id for TRL→reset) ---
def make_stage_dataset(num_episodes: int, task_ids: list) -> Dataset:
    return build_grpo_dataset(int(num_episodes), list(task_ids))

dataset = make_stage_dataset(NUM_EPISODES, CURRICULUM_STAGES[0]['task_ids'])

# --- 6. Debug print: confirm chat template + tool list reach the model ---
# Many "0 tool calls" episodes were caused by the chat template silently
# wrapping completions in <think>...</think> blocks. Lock enable_thinking
# off and print one full prompt so we can eyeball the formatting.
_dbg_prompt = dataset[0]['prompt']
try:
    debug_prompt = tokenizer.apply_chat_template(
        _dbg_prompt,
        add_generation_prompt=True,
        tokenize=False,
        enable_thinking=False,
    )
except TypeError:
    debug_prompt = tokenizer.apply_chat_template(
        _dbg_prompt,
        add_generation_prompt=True,
        tokenize=False,
    )
print('=== DEBUG: rendered chat-template prompt (first 1200 chars) ===')
print(debug_prompt[:1200])
print('=== END DEBUG ===')

# --- 7. Load model with 4-bit quantization (T4-safe) --------------------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)

# --- 8. GRPO Config ------------------------------------------------------
grpo_kwargs = dict(
    use_vllm=False,
    output_dir=str(output_dir),
    num_train_epochs=1,
    learning_rate=2e-6,
    lr_scheduler_type='cosine',
    max_tool_calling_iterations=MAX_TURNS,
    warmup_steps=2,
    max_grad_norm=1.0,
    gradient_accumulation_steps=1,
    per_device_train_batch_size=1,
    generation_batch_size=NUM_GENERATIONS,
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_NEW_TOKENS,
    logging_steps=1,
    save_strategy='steps',
    save_steps=10,
    temperature=TEMPERATURE,
    report_to='trackio',
    run_name=f'pagezero-grpo-{timestamp}',
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    save_total_limit=3,
    loss_type='dapo',
    mask_truncated_completions=True,
    beta=KL_BETA,
    # Critical: disable Qwen3 chain-of-thought wrapper so completions parse
    # to <tool_call> blocks instead of <think>...</think> blobs.
    chat_template_kwargs={'enable_thinking': False},
)

grpo_fields = set(getattr(GRPOConfig, '__dataclass_fields__', {}).keys())
if 'use_liger_kernel' in grpo_fields:
    grpo_kwargs['use_liger_kernel'] = True
if 'project' in grpo_fields:
    grpo_kwargs['project'] = 'pagezero-rl'
if 'top_p' in grpo_fields:
    grpo_kwargs['top_p'] = TOP_P

grpo_config = GRPOConfig(**grpo_kwargs)

# --- 9. LoRA Config ------------------------------------------------------
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)

# --- 10. Initialize Trainer (single primary reward) ---------------------
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_total_audited],
    train_dataset=dataset,
    args=grpo_config,
    peft_config=peft_config,
    environment_factory=PageZeroToolEnv,
)

print(f'Trainer initialized. Output path: {output_dir}')
print(f'  KL β             : {KL_BETA}')
print(f'  num_generations  : {NUM_GENERATIONS}  (g)')
print(f'  temperature      : {TEMPERATURE}')
print(f'  top_p            : {TOP_P} (if supported by GRPOConfig)')
print(f'  max_completion_length: {MAX_NEW_TOKENS}')
print(f'  mask_truncated   : False (Tier 1 — keep truncated rollouts in the loss)')
print(f'  enable_thinking  : False (forced by chat_template_kwargs)')
print(f'  reward_funcs     : [reward_total_audited] (single primary)')
print(f'  curriculum stages: {[s["name"] for s in CURRICULUM_STAGES]}')
print(f'  reward log       : {reward_logger.csv_path}')
print(f'  trajectory log   : {reward_logger.jsonl_path}')




print('Starting GRPO curriculum training (mastery-gated)...')
print(f'  Model        : {MODEL_ID}')
print(f'  KL β         : {KL_BETA}')
print(f'  g (num_gen)  : {NUM_GENERATIONS}')
print(f'  Episodes/stg : {NUM_EPISODES} (upper bound; gate may end early)')
print(f'  Environment  : {ENV_URL}')
print(f'  Stages       : {[s["name"] for s in CURRICULUM_STAGES]}')
print()


# --- Curriculum loop -----------------------------------------------------
# The MasteryCurriculum tracks recent_success_rate over a sliding window;
# advancement happens *inside* the wrapper as soon as recent_success_rate
# clears the tier's advance_rate. Here we simply iterate over the stages
# and let the trainer run up to NUM_EPISODES rollouts per stage.
STAGE_CHECKPOINTS: dict[str, "Path"] = {}
STAGE_SUMMARIES: dict[str, dict] = {}


def _summarize(jsonl_path, stage):
    """Aggregate trajectory-level stats for one stage from the global JSONL."""
    if not jsonl_path.exists():
        return {}
    rows = []
    with open(jsonl_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
            except Exception:
                continue
            if row.get('stage') == stage:
                rows.append(row)
    if not rows:
        return {}
    totals = [float(r.get('total_reward', 0.0)) for r in rows]
    norms = [float(r.get('normalized_reward', 0.0)) for r in rows]
    resolved = [bool(r.get('is_resolved')) for r in rows]
    n_steps = [int(r.get('num_steps', 0)) for r in rows]
    tool_freq = {}
    for r in rows:
        for t in r.get('tool_sequence', []):
            tool_freq[t] = tool_freq.get(t, 0) + 1
    return {
        'episodes': len(rows),
        'mean_reward': sum(totals) / len(totals),
        'mean_normalized': sum(norms) / len(norms),
        'best_reward': max(totals),
        'worst_reward': min(totals),
        'resolution_rate': sum(resolved) / len(resolved),
        'mean_steps': sum(n_steps) / len(n_steps),
        'top_tools': sorted(tool_freq.items(), key=lambda kv: -kv[1])[:5],
    }


try:
    for stage in CURRICULUM_STAGES:
        stage_name = stage['name']
        # Bind the stage so the wrapper samples task ids from the right pool
        # (the curriculum may *also* bias toward weak spots inside that pool).
        PageZeroToolEnv._set_stage(stage_name, stage['task_ids'])

        # Refresh dataset for this stage; reset trainer epoch state so
        # repeated trainer.train() calls behave like clean stages.
        trainer.train_dataset = make_stage_dataset(stage['episodes'], stage['task_ids'])
        stage_dir = output_dir / f'stage_{stage_name}'
        stage_dir.mkdir(parents=True, exist_ok=True)
        trainer.args.output_dir = str(stage_dir)
        trainer.args.run_name = f'pagezero-grpo-{timestamp}-{stage_name}'
        trainer.state.epoch = 0
        if hasattr(trainer.state, 'global_step'):
            trainer.state.global_step = 0
        if hasattr(trainer, '_episodes_completed'):
            trainer._episodes_completed = 0

        print('=' * 70)
        print(f'[curriculum] STAGE = {stage_name.upper()}  '
              f'(episodes_budget={stage["episodes"]}, β={KL_BETA}, '
              f'g={NUM_GENERATIONS}, tasks={stage["task_ids"]})')
        print(f'[curriculum] tier={mastery_curriculum.get_tier_name()} '
              f'recent_success_rate={mastery_curriculum._recent_success_rate():.0%}')
        print('=' * 70)

        # Initialize trackio for each training run to ensure proper logging
        trackio.init(project = "openenv")

        try:
            trainer.train()
        finally:
            PageZeroToolEnv._close_all()

        # Save a per-stage transition checkpoint for rollback / comparison.
        ckpt_dir = stage_dir / 'checkpoint'
        ckpt_dir.mkdir(parents=True, exist_ok=True)
        trainer.save_model(str(ckpt_dir))
        STAGE_CHECKPOINTS[stage_name] = ckpt_dir
        print(f'[curriculum] {stage_name} → checkpoint saved at {ckpt_dir}')

        summary = _summarize(reward_logger.jsonl_path, stage_name)
        STAGE_SUMMARIES[stage_name] = summary
        if summary:
            print(
                f'[curriculum] {stage_name} summary: '
                f'episodes={summary["episodes"]} '
                f'mean={summary["mean_reward"]:.3f} '
                f'normalized={summary["mean_normalized"]:.3f} '
                f'best={summary["best_reward"]:.3f} '
                f'resolved={summary["resolution_rate"]:.0%} '
                f'avg_steps={summary["mean_steps"]:.1f} '
                f'tier_after={mastery_curriculum.get_tier_name()}'
            )

        gc.collect()
        torch.cuda.empty_cache()

finally:
    PageZeroToolEnv._close_all()


# --- Final post-curriculum save + reporting -----------------------------
final_dir = output_dir / 'final'
final_dir.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(final_dir))
print(f'\nFinal model saved to {final_dir}')

# Single canonical reward plot from the top-level CSV.
try:
    plot_rewards(reward_logger.csv_path, output_dir / 'reward_plot.png')
    if (output_dir / 'reward_plot.png').exists():
        display(Image(filename=str(output_dir / 'reward_plot.png')))
except Exception as e:
    print(f'WARNING: could not plot rewards: {e}')

print('\n=== Curriculum stage checkpoints ===')
for name, path in STAGE_CHECKPOINTS.items():
    print(f'  {name:6s} → {path}')

print('\n=== Per-stage trajectory summary ===')
summary_rows = []
for name, summary in STAGE_SUMMARIES.items():
    if not summary:
        continue
    summary_rows.append({
        'stage': name,
        'episodes': summary['episodes'],
        'mean_reward': round(summary['mean_reward'], 3),
        'normalized_mean': round(summary['mean_normalized'], 3),
        'best_reward': round(summary['best_reward'], 3),
        'worst_reward': round(summary['worst_reward'], 3),
        'resolution_rate': round(summary['resolution_rate'], 3),
        'mean_steps': round(summary['mean_steps'], 2),
    })
if summary_rows:
    display(pd.DataFrame(summary_rows))

# Show last N rows of the canonical reward log for a quick sanity check.
if reward_logger.csv_path.exists():
    df = pd.read_csv(reward_logger.csv_path)
    display(df.tail(15))

if PUSH_TO_HUB:
    if 'HF_TOKEN' not in os.environ:
        raise RuntimeError('HF_TOKEN not set. Add it in Colab secrets or env vars.')
    trainer.push_to_hub(repo_id=HUB_REPO)
    print(f'Model pushed to https://huggingface.co/{HUB_REPO}')
else:
    print('Skipping push. Set PUSH_TO_HUB=True to enable.')


Output dir       : outputs/pagezero-colab-2026-04-26_08-40-14
Reward log       : outputs/pagezero-colab-2026-04-26_08-40-14/reward_log.csv
Trajectory log   : outputs/pagezero-colab-2026-04-26_08-40-14/trajectories.jsonl
Curriculum tier  : warmup
=== DEBUG: rendered chat-template prompt (first 1200 chars) ===
<|im_start|>system
You are a Staff SRE on-call. Diagnose and fix the cascading incident across the Application, PostgreSQL, and Redis cache.

To use a tool, you MUST use this exact format:
<tool_call>
{"name": "check_alerts", "arguments": {}}
</tool_call>

## Tool cheat-sheet (when to use what)
- **Triage (first step almost always here):** `check_alerts` — container/app health snapshot. Do not skip for generic “what is on fire?” context.
- **Postgres CPU / latency / connections:** prefer `pg_stat_activity`, `pg_locks`, `pg_stat_statements`, `pg_explain_analyze` before any fix. Fixes: `pg_cancel_query`, `pg_create_index`, `pg_vacuum`.
- **Redis memory / OOM / cache:** prefer `redis_

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

/tmp/ipykernel_5512/2009504917.py:182: UserWarning: You are using 'environment_factory', which is an experimental feature. This API may change or be removed at any time without prior notice. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  trainer = GRPOTrainer(
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Trainer initialized. Output path: outputs/pagezero-colab-2026-04-26_08-40-14
  KL β             : 0.01
  num_generations  : 4  (g)
  temperature      : 0.6
  top_p            : 0.9 (if supported by GRPOConfig)
  max_completion_length: 512
  mask_truncated   : False (Tier 1 — keep truncated rollouts in the loss)
  enable_thinking  : False (forced by chat_template_kwargs)
  reward_funcs     : [reward_total_audited] (single primary)
  curriculum stages: ['easy', 'medium', 'hard']
  reward log       : outputs/pagezero-colab-2026-04-26_08-40-14/reward_log.csv
  trajectory log   : outputs/pagezero-colab-2026-04-26_08-40-14/trajectories.jsonl
Starting GRPO curriculum training (mastery-gated)...
  Model        : Qwen/Qwen3-0.6B
  KL β         : 0.01
  g (num_gen)  : 4
  Episodes/stg : 4 (upper bound; gate may end early)
  Environment  : https://neokazuha-pagezero-agent.hf.space
  Stages       : ['easy', 'medium', 'hard']

[curriculum] STAGE = EASY  (episodes_budget=4, β=0.01, g=4, tasks=['task

[nb-reset] stage=easy row_task=task_1 chosen=task_1
[nb-reset] stage=easy row_task=task_1 chosen=task_1
[nb-reset] stage=easy row_task=task_1 chosen=task_1
[nb-reset] stage=easy row_task=task_1 chosen=task_1
  [step  1] DIAG check_alerts           r=+0.200 cum=+0.200
  [step  1] DIAG check_alerts           r=+0.200 cum=+0.200
  [step  1] DIAG check_alerts           r=+0.200 cum=+0.200
  [step  1] DIAG pg_stat_activity       r=-0.300 cum=-0.300
  [step  2] DIAG pg_stat_activity       r=+0.050 cum=+0.250
  [step  2] DIAG pg_stat_activity       r=+0.050 cum=+0.250
  [step  2] DIAG pg_stat_activity       r=+0.050 cum=+0.250
  [step  2] DIAG pg_stat_activity       r=-0.350 cum=-0.650
  [reward-audit:total] n=4 mean=+0.025 std=0.450 min=-0.650 max=+0.250


Step,Training Loss
1,-0.497611
2,-0.499275
3,-0.500684
4,1.498484
5,0.500442
6,0.502934
7,-1.500020


[nb-reset] stage=easy row_task=task_1 chosen=task_1
[nb-reset] stage=easy row_task=task_1 chosen=task_1
[nb-reset] stage=easy row_task=task_1 chosen=task_1
[nb-reset] stage=easy row_task=task_1 chosen=task_1
  [step  1] DIAG pg_stat_activity       r=-0.300 cum=-0.300
  [step  1] DIAG check_alerts           r=+0.200 cum=+0.200
  [step  1] DIAG pg_stat_activity       r=-0.300 cum=-0.300
  [step  1] DIAG pg_stat_activity       r=-0.300 cum=-0.300
  [step  2] DIAG pg_stat_activity       r=+0.050 cum=+0.250
  [step  2] DIAG pg_stat_activity       r=-0.350 cum=-0.650
  [step  2] DIAG pg_stat_activity       r=-0.350 cum=-0.650
  [step  2] DIAG pg_stat_activity       r=-0.350 cum=-0.650
  [reward-audit:total] n=4 mean=-0.425 std=0.450 min=-0.650 max=+0.250
[nb-reset] stage=easy row_task=task_1 chosen=task_1
[nb-reset] stage=easy row_task=task_1 chosen=task_1
[nb-reset] stage=easy row_task=task_1 chosen=task_1
[nb-reset] stage=easy row_task=task_1 chosen=task_1
  [step  1] DIAG check_alerts    

KeyboardInterrupt: 

## 7.5) Patches: Deterministic Task Binding

Runtime monkey-patches that make the dataset row's `episode_task_id` strictly bind to the env reset, and enforce `NUM_EPISODES` divisibility by `NUM_GENERATIONS` so GRPO comparison groups stay coherent.

**Must run after Section 7 (`PageZeroToolEnv` + `make_stage_dataset` are defined) and before Section 8 (training).**


In [8]:
# NOTE: Run this cell after the notebook class/function definitions are loaded.
# It enforces deterministic dataset->env task binding and group-size integrity.

def _patch_notebook_for_deterministic_tasks():
    import types

    def _reset_strict(self, **kwargs):
        kwargs.pop('prompt', None)
        row_tid = kwargs.pop('episode_task_id', None)
        row_tid_s = str(row_tid).strip() if row_tid is not None else ''
        pool = [str(x) for x in (self.STAGE_TASK_IDS or [])]

        chosen_task_id = ''
        if row_tid_s and pool and row_tid_s in pool:
            chosen_task_id = row_tid_s
        elif row_tid_s and pool:
            raise ValueError(f"episode_task_id={row_tid_s!r} not in active stage pool={pool}")
        elif not row_tid_s and pool:
            raise ValueError('episode_task_id missing from dataset row; deterministic grouping requires it.')
        elif self.CURRICULUM is not None:
            chosen_task_id = self.CURRICULUM.pick_task_id(self.STAGE)
        elif self.STAGE_TASK_IDS:
            import random
            chosen_task_id = random.choice(self.STAGE_TASK_IDS)

        if self._client is not None:
            try:
                _sync(self._client.close(), timeout=10)
            except Exception:
                pass
            self._client = None
            time.sleep(0.3)
        self._client = PageZeroEnvClient(base_url=self.BASE_URL, message_timeout_s=300.0)

        try:
            result = _sync(self._client.reset(task_id=chosen_task_id)) if chosen_task_id else _sync(self._client.reset())
        except TypeError:
            result = _sync(self._client.reset())
            chosen_task_id = ''

        self.total_reward = 0.0
        self.diagnosis_reward = 0.0
        self.fix_reward = 0.0
        self.terminal_reward = 0.0
        self.is_done = bool(result.done)
        self._episode_logged = False
        self.trajectory = []
        self.task_id = chosen_task_id
        self.scenario_name = (getattr(result.observation, 'tool_output', '') or '')[:120]
        self.last_sla_status = getattr(result.observation, 'sla_status', 'OK') or 'OK'
        self.is_resolved = False
        self.stack_healthy = None
        self.start_ts = time.time()
        self.end_ts = None
        self._last_feedback = ''
        self._last_command = ''
        self._last_output_snippet = ''
        self._last_reward = 0.0

        print(f"[nb-reset] stage={self.STAGE} row_task={row_tid_s or '<missing>'} chosen={chosen_task_id or '<auto>'}")
        return self._format_observation(result.observation, reward=0.0)

    def _make_stage_dataset_strict(num_episodes: int, task_ids: list) -> Dataset:
        if int(num_episodes) < int(NUM_GENERATIONS):
            raise ValueError(
                f'NUM_EPISODES must be >= NUM_GENERATIONS; got {num_episodes} and {NUM_GENERATIONS}'
            )
        if int(num_episodes) % int(NUM_GENERATIONS) != 0:
            raise ValueError(
                f'NUM_EPISODES must be divisible by NUM_GENERATIONS; got {num_episodes} and {NUM_GENERATIONS}'
            )
        return build_grpo_dataset(int(num_episodes), list(task_ids), group_size=int(NUM_GENERATIONS))

    _orig_run_tool = PageZeroToolEnv._run_tool

    def _run_tool_with_gate_debug(self, tool, args):
        text = _orig_run_tool(self, tool, args)
        fb = str(getattr(self, '_last_feedback', '') or '')
        if 'Ignored `done`' in fb or 'documentation incomplete' in fb or 'deferred until step >=' in fb:
            print(f"  [gate-debug] {fb[:220]}")
        return text

    PageZeroToolEnv.reset = _reset_strict
    PageZeroToolEnv._run_tool = _run_tool_with_gate_debug
    globals()['make_stage_dataset'] = _make_stage_dataset_strict
    print('[patch] deterministic task binding + generation-group integrity + done/docs gate debug enabled')


_patch_notebook_for_deterministic_tasks()


[patch] deterministic task binding + generation-group integrity + done/docs gate debug enabled


## 8) Train, Plot, and Save

In [9]:
PUSH_TO_HUB = True

In [11]:
import json, gc
import pandas as pd
from IPython.display import display, Image
from PageZero.train import plot_rewards

print('Starting GRPO curriculum training (mastery-gated)...')
print(f'  Model        : {MODEL_ID}')
print(f'  KL β         : {KL_BETA}')
print(f'  g (num_gen)  : {NUM_GENERATIONS}')
print(f'  Episodes/stg : {NUM_EPISODES} (upper bound; gate may end early)')
print(f'  Environment  : {ENV_URL}')
print(f'  Stages       : {[s["name"] for s in CURRICULUM_STAGES]}')
print()


# --- Curriculum loop -----------------------------------------------------
# The MasteryCurriculum tracks recent_success_rate over a sliding window;
# advancement happens *inside* the wrapper as soon as recent_success_rate
# clears the tier's advance_rate. Here we simply iterate over the stages
# and let the trainer run up to NUM_EPISODES rollouts per stage.
STAGE_CHECKPOINTS: dict[str, "Path"] = {}
STAGE_SUMMARIES: dict[str, dict] = {}


def _summarize(jsonl_path, stage):
    """Aggregate trajectory-level stats for one stage from the global JSONL."""
    if not jsonl_path.exists():
        return {}
    rows = []
    with open(jsonl_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
            except Exception:
                continue
            if row.get('stage') == stage:
                rows.append(row)
    if not rows:
        return {}
    totals = [float(r.get('total_reward', 0.0)) for r in rows]
    norms = [float(r.get('normalized_reward', 0.0)) for r in rows]
    resolved = [bool(r.get('is_resolved')) for r in rows]
    n_steps = [int(r.get('num_steps', 0)) for r in rows]
    tool_freq = {}
    for r in rows:
        for t in r.get('tool_sequence', []):
            tool_freq[t] = tool_freq.get(t, 0) + 1
    return {
        'episodes': len(rows),
        'mean_reward': sum(totals) / len(totals),
        'mean_normalized': sum(norms) / len(norms),
        'best_reward': max(totals),
        'worst_reward': min(totals),
        'resolution_rate': sum(resolved) / len(resolved),
        'mean_steps': sum(n_steps) / len(n_steps),
        'top_tools': sorted(tool_freq.items(), key=lambda kv: -kv[1])[:5],
    }


try:
    for stage in CURRICULUM_STAGES:
        stage_name = stage['name']
        # Bind the stage so the wrapper samples task ids from the right pool
        # (the curriculum may *also* bias toward weak spots inside that pool).
        PageZeroToolEnv._set_stage(stage_name, stage['task_ids'])

        # Refresh dataset for this stage; reset trainer epoch state so
        # repeated trainer.train() calls behave like clean stages.
        trainer.train_dataset = make_stage_dataset(stage['episodes'], stage['task_ids'])
        stage_dir = output_dir / f'stage_{stage_name}'
        stage_dir.mkdir(parents=True, exist_ok=True)
        trainer.args.output_dir = str(stage_dir)
        trainer.args.run_name = f'pagezero-grpo-{timestamp}-{stage_name}'
        trainer.state.epoch = 0
        if hasattr(trainer.state, 'global_step'):
            trainer.state.global_step = 0
        if hasattr(trainer, '_episodes_completed'):
            trainer._episodes_completed = 0

        print('=' * 70)
        print(f'[curriculum] STAGE = {stage_name.upper()}  '
              f'(episodes_budget={stage["episodes"]}, β={KL_BETA}, '
              f'g={NUM_GENERATIONS}, tasks={stage["task_ids"]})')
        print(f'[curriculum] tier={mastery_curriculum.get_tier_name()} '
              f'recent_success_rate={mastery_curriculum._recent_success_rate():.0%}')
        print('=' * 70)

        try:
            trainer.train()
        finally:
            PageZeroToolEnv._close_all()

        # Save a per-stage transition checkpoint for rollback / comparison.
        ckpt_dir = stage_dir / 'checkpoint'
        ckpt_dir.mkdir(parents=True, exist_ok=True)
        trainer.save_model(str(ckpt_dir))
        STAGE_CHECKPOINTS[stage_name] = ckpt_dir
        print(f'[curriculum] {stage_name} → checkpoint saved at {ckpt_dir}')

        summary = _summarize(reward_logger.jsonl_path, stage_name)
        STAGE_SUMMARIES[stage_name] = summary
        if summary:
            print(
                f'[curriculum] {stage_name} summary: '
                f'episodes={summary["episodes"]} '
                f'mean={summary["mean_reward"]:.3f} '
                f'normalized={summary["mean_normalized"]:.3f} '
                f'best={summary["best_reward"]:.3f} '
                f'resolved={summary["resolution_rate"]:.0%} '
                f'avg_steps={summary["mean_steps"]:.1f} '
                f'tier_after={mastery_curriculum.get_tier_name()}'
            )

        gc.collect()
        torch.cuda.empty_cache()

finally:
    PageZeroToolEnv._close_all()


# --- Final post-curriculum save + reporting -----------------------------
final_dir = output_dir / 'final'
final_dir.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(final_dir))
print(f'\nFinal model saved to {final_dir}')

# Single canonical reward plot from the top-level CSV.
try:
    plot_rewards(reward_logger.csv_path, output_dir / 'reward_plot.png')
    if (output_dir / 'reward_plot.png').exists():
        display(Image(filename=str(output_dir / 'reward_plot.png')))
except Exception as e:
    print(f'WARNING: could not plot rewards: {e}')

print('\n=== Curriculum stage checkpoints ===')
for name, path in STAGE_CHECKPOINTS.items():
    print(f'  {name:6s} → {path}')

print('\n=== Per-stage trajectory summary ===')
summary_rows = []
for name, summary in STAGE_SUMMARIES.items():
    if not summary:
        continue
    summary_rows.append({
        'stage': name,
        'episodes': summary['episodes'],
        'mean_reward': round(summary['mean_reward'], 3),
        'normalized_mean': round(summary['mean_normalized'], 3),
        'best_reward': round(summary['best_reward'], 3),
        'worst_reward': round(summary['worst_reward'], 3),
        'resolution_rate': round(summary['resolution_rate'], 3),
        'mean_steps': round(summary['mean_steps'], 2),
    })
if summary_rows:
    display(pd.DataFrame(summary_rows))

# Show last N rows of the canonical reward log for a quick sanity check.
if reward_logger.csv_path.exists():
    df = pd.read_csv(reward_logger.csv_path)
    display(df.tail(15))

if PUSH_TO_HUB:
    if 'HF_TOKEN' not in os.environ:
        raise RuntimeError('HF_TOKEN not set. Add it in Colab secrets or env vars.')
    trainer.push_to_hub(repo_id=HUB_REPO)
    print(f'Model pushed to https://huggingface.co/{HUB_REPO}')
else:
    print('Skipping push. Set PUSH_TO_HUB=True to enable.')


Starting GRPO curriculum training (mastery-gated)...
  Model        : Qwen/Qwen3-0.6B
  KL β         : 0.01
  g (num_gen)  : 4
  Episodes/stg : 4 (upper bound; gate may end early)
  Environment  : https://neokazuha-pagezero-agent.hf.space
  Stages       : ['easy', 'medium', 'hard']

[curriculum] STAGE = EASY  (episodes_budget=4, β=0.01, g=4, tasks=['task_1', 'task_2', 'task_3', 'task_4', 'task_5'])
[curriculum] tier=warmup recent_success_rate=0%


Step,Training Loss


RuntimeError: Call trackio.init() before trackio.log().

## 9) Adversarial Self-Play (Red Breaker vs Blue Fixer)

This section adds asymmetric self-play on top of the existing GRPO setup:

- **Red agent (breaker):** picks the most damaging task/scenario for Blue.
- **Blue agent (fixer):** the GRPO policy that diagnoses and resolves incidents.
- **Red reward:** increases with unresolved incidents, downtime, and revenue loss.
- **Blue reward:** unchanged (existing environment reward pipeline).

If `GEMINI_API_KEY` is set, Red task selection is delegated to Gemini. Otherwise a deterministic heuristic fallback is used.

In [16]:
import json
import random
from dataclasses import dataclass
from typing import Dict, List, Any, Optional
import requests

# -------------------- Adversarial config --------------------
ENABLE_SELF_PLAY = True

# Number of Red→Blue alternation rounds per curriculum stage.
# Tighter alternation (fewer rounds, more stages) improves convergence.
SELF_PLAY_ROUNDS_PER_STAGE = 2

RED_FAULT_BUDGET = 1

# Red reward weights (deterministic component from episode outcomes)
RED_REWARD_ALPHA_UNRESOLVED = 1.2   # +1.2 when Blue fails to resolve
RED_REWARD_ALPHA_DOWNTIME = 0.25    # +0.25 per minute of downtime
RED_REWARD_ALPHA_REVENUE = 0.02     # +0.02 per dollar of revenue loss
RED_REWARD_ALPHA_STEPS = 0.03       # +0.03 per step Blue takes (longer = better for Red)

# Gemini reward for Red: ask Gemini to rate how damaging Red's choice was.
# Blended with deterministic: GEMINI_RED_WEIGHT * gemini + (1-weight) * det
USE_GEMINI_RED_AGENT = bool(os.environ.get('GEMINI_API_KEY', '').strip())
GEMINI_MODEL = os.environ.get('GEMINI_MODEL', 'gemini-2.5-flash')
GEMINI_RED_WEIGHT = 0.4   # how much Gemini reward contributes to Red's total

print(f'[adv-config] USE_GEMINI_RED_AGENT: {USE_GEMINI_RED_AGENT}')
print(f'[adv-config] GEMINI_MODEL: {GEMINI_MODEL}')
print(f'[adv-config] GEMINI_RED_WEIGHT: {GEMINI_RED_WEIGHT}')
print(f'[adv-config] SELF_PLAY_ROUNDS_PER_STAGE: {SELF_PLAY_ROUNDS_PER_STAGE}')


@dataclass
class RedAction:
    stage: str
    task_id: str
    rationale: str
    target_failure_mode: str


def _gemini_post(prompt_text: str, timeout: int = 30) -> Optional[dict]:
    """POST a prompt to Gemini and return parsed JSON or None on failure."""
    api_key = os.environ.get('GEMINI_API_KEY', '').strip()
    if not api_key:
        return None
    url = (
        'https://generativelanguage.googleapis.com/v1beta/models/'
        f'{GEMINI_MODEL}:generateContent?key={api_key}'
    )
    body = {
        'contents': [{'role': 'user', 'parts': [{'text': prompt_text}]}],
        'generationConfig': {'temperature': 0.1, 'responseMimeType': 'application/json'},
    }
    try:
        r = requests.post(url, json=body, timeout=timeout)
        r.raise_for_status()
        text = r.json()['candidates'][0]['content']['parts'][0]['text']
        return json.loads(text)
    except Exception as e:
        print(f'[gemini_post] FAILED: {e}')
        return None


class RedBreakerAgent:
    """Adversarial task selector with Gemini-evaluated reward signal.

    Task selection: Gemini (if available) → heuristic fallback.
    Red reward: deterministic outcome score blended with Gemini quality score.
    Learns online from episode stats to prioritize the hardest tasks.
    """

    def __init__(self):
        self.task_stats: Dict[str, Dict[str, float]] = {}
        self._gemini_select_calls = 0
        self._gemini_select_hits = 0
        self._gemini_reward_calls = 0
        self._gemini_reward_hits = 0

    def debug_stats(self) -> str:
        return (
            f'[red-agent] select calls={self._gemini_select_calls} '
            f'hits={self._gemini_select_hits} | '
            f'reward calls={self._gemini_reward_calls} '
            f'hits={self._gemini_reward_hits}'
        )

    def _score_task(self, task_id: str) -> float:
        """Heuristic task score: higher = more adversarial."""
        s = self.task_stats.get(task_id, {})
        success = float(s.get('blue_success_rate', 0.5))
        avg_downtime = float(s.get('avg_downtime', 0.0))
        avg_red = float(s.get('avg_red_reward', 0.0))
        return (1.0 - success) * 1.8 + avg_downtime * 0.08 + avg_red * 0.4

    def _gemini_choose(self, stage_name: str, task_ids: List[str]) -> Optional[RedAction]:
        if not os.environ.get('GEMINI_API_KEY', '').strip():
            return None

        self._gemini_select_calls += 1
        ranked = sorted(task_ids, key=self._score_task, reverse=True)
        stats_view = {
            t: self.task_stats.get(t, {
                'blue_success_rate': 0.5, 'avg_downtime': 0.0,
                'avg_red_reward': 0.0, 'episodes': 0,
            })
            for t in ranked
        }

        prompt = json.dumps({
            'task': 'You are a Red adversarial agent. Choose the ONE task_id that is maximally ',
            'stage': stage_name,
            'budget': RED_FAULT_BUDGET,
            'task_ids': task_ids,
            'task_stats': stats_view,
            'objective': (
                'You are a Red adversarial agent. Choose the ONE task_id that is maximally '
                'adversarial to a Blue SRE fixer. Consider: low blue success rate, high downtime, '
                'cascading failure potential. Return JSON with keys: task_id, rationale, target_failure_mode.'
            ),
            'constraints': [
                'task_id must be exactly one of the provided task_ids',
                'rationale <= 30 words',
                'target_failure_mode: one of latency, oom, lock, cascade, disk, permission',
            ],
        })

        obj = _gemini_post(prompt)
        if obj is None:
            return None

        chosen = obj.get('task_id', '').strip()
        if chosen not in task_ids:
            print(f'[red-agent/gemini] Returned invalid task_id={chosen!r}, fallback')
            return None

        self._gemini_select_hits += 1
        print(
            f'[red-agent/gemini] Selected task_id={chosen} '
            f'rationale="{obj.get("rationale", "")}" '
            f'failure_mode={obj.get("target_failure_mode", "?")}'
        )
        return RedAction(
            stage=stage_name,
            task_id=chosen,
            rationale=obj.get('rationale', 'gemini adversarial pick'),
            target_failure_mode=obj.get('target_failure_mode', 'unknown'),
        )

    def choose(self, stage_name: str, task_ids: List[str]) -> RedAction:
        if USE_GEMINI_RED_AGENT:
            pick = self._gemini_choose(stage_name, task_ids)
            if pick is not None:
                return pick

        ranked = sorted(task_ids, key=self._score_task, reverse=True)
        chosen = ranked[0] if ranked else random.choice(task_ids)
        print(
            f'[red-agent/heuristic] Selected task_id={chosen} '
            f'(score={self._score_task(chosen):.3f})'
        )
        return RedAction(
            stage=stage_name,
            task_id=chosen,
            rationale='heuristic: maximize unresolved+downtime likelihood',
            target_failure_mode='latency/cascading-failure',
        )

    def gemini_evaluate_red_reward(
        self,
        task_id: str,
        episodes: List[dict],
        det_red_reward: float,
    ) -> float:
        """Ask Gemini to rate how damaging Red's choice was for Blue.

        Blended with deterministic outcome reward: GEMINI_RED_WEIGHT * gemini + rest * det.
        Returns final blended red reward. Falls back to det_red_reward if Gemini unavailable.
        """
        self._gemini_reward_calls += 1
        if not os.environ.get('GEMINI_API_KEY', '').strip() or not episodes:
            print(
                f'[red-agent/reward] det_red={det_red_reward:.3f} '
                f'(Gemini skipped — no key or empty episodes)'
            )
            return det_red_reward

        summary = [{
            'is_resolved': ep.get('is_resolved'),
            'num_steps': ep.get('num_steps'),
            'downtime_minutes': ep.get('downtime_minutes', 0.0),
            'revenue_loss_usd': ep.get('revenue_loss_usd', 0.0),
            'blue_total_reward': ep.get('total_reward', 0.0),
            'tool_sequence': (ep.get('tool_sequence') or [])[:8],
        } for ep in episodes[:4]]

        prompt = json.dumps({
            'task': 'Rate how damaging this Red adversarial task was for the Blue SRE fixer.',
            'task_id': task_id,
            'blue_episode_summaries': summary,
            'scoring_guide': {
                '1.0': 'Blue completely failed, high downtime, no resolution',
                '0.7': 'Blue partially failed or took very long',
                '0.4': 'Blue succeeded but struggled',
                '0.1': 'Blue resolved quickly — Red choice was not effective',
            },
            'required_output': {'gemini_red_score': 'float 0.0-1.0', 'reasoning': 'string <= 30 words'},
        })

        obj = _gemini_post(prompt, timeout=30)
        if obj is None:
            print(
                f'[red-agent/reward] det_red={det_red_reward:.3f} '
                f'(Gemini reward call failed → using det)'
            )
            return det_red_reward

        gemini_score = float(obj.get('gemini_red_score', 0.5))
        # Map [0,1] → reward-space [0, RED_REWARD_ALPHA_UNRESOLVED * 1.5] for comparability
        gemini_reward = gemini_score * RED_REWARD_ALPHA_UNRESOLVED * 1.5
        blended = GEMINI_RED_WEIGHT * gemini_reward + (1.0 - GEMINI_RED_WEIGHT) * det_red_reward
        self._gemini_reward_hits += 1

        print(
            f'[red-agent/reward] task={task_id} '
            f'gemini_score={gemini_score:.3f} gemini_r={gemini_reward:.3f} '
            f'det_r={det_red_reward:.3f} blended={blended:.3f} '
            f'reasoning="{obj.get("reasoning", "")}"'
        )
        return blended

    def update(self, task_id: str, episodes: List[dict], red_reward: Optional[float] = None) -> None:
        if not episodes:
            return
        s = self.task_stats.setdefault(task_id, {
            'episodes': 0, 'blue_success_rate': 0.5,
            'avg_downtime': 0.0, 'avg_red_reward': 0.0,
        })
        n0 = float(s['episodes'])
        n = n0 + len(episodes)
        successes = sum(1 for ep in episodes if bool(ep.get('is_resolved')))
        mean_success = successes / len(episodes)
        mean_downtime = (
            sum(float(ep.get('downtime_minutes', 0.0)) for ep in episodes) / len(episodes)
        )
        mean_red = red_reward if red_reward is not None else (
            sum(float(ep.get('red_reward', 0.0)) for ep in episodes) / len(episodes)
        )
        s['blue_success_rate'] = (s['blue_success_rate'] * n0 + mean_success * len(episodes)) / n
        s['avg_downtime'] = (s['avg_downtime'] * n0 + mean_downtime * len(episodes)) / n
        s['avg_red_reward'] = (s['avg_red_reward'] * n0 + mean_red * len(episodes)) / n
        s['episodes'] = int(n)
        print(
            f'[red-agent/update] task={task_id} '
            f'blue_success={s["blue_success_rate"]:.2f} '
            f'avg_downtime={s["avg_downtime"]:.1f}m '
            f'avg_red_r={s["avg_red_reward"]:.3f} '
            f'total_eps={s["episodes"]}'
        )


class AdversarialPageZeroToolEnv(PageZeroToolEnv):
    """Drop-in env wrapper with Red task override + red reward accounting.

    Forces Blue to face the exact task chosen by Red (bypassing the
    MasteryCurriculum, which would otherwise resample inside the stage
    pool) and augments ``trajectory_payload`` with Red-side metrics so
    they land in the per-stage trajectories.jsonl.

    Section 10 also disables ``CURRICULUM`` for the duration of the
    adversarial run so curriculum mastery stats are not polluted by
    adversarial outliers.
    """

    RED_TASK_OVERRIDE: Optional[str] = None
    RED_METADATA: Dict[str, Any] = {}

    @classmethod
    def set_red_attack(cls, task_id: Optional[str], metadata: Optional[Dict[str, Any]] = None):
        """Sets the Red agent's chosen attack task and associated metadata.

        Args:
            task_id: The ID of the task chosen by the Red agent.
            metadata: A dictionary containing additional metadata about Red's attack.
        """
        cls.RED_TASK_OVERRIDE = task_id
        cls.RED_METADATA = metadata or {}

    def reset(self, **kwargs) -> str | None:
        # Force the canonical wrapper's reset path to use STAGE_TASK_IDS
        # (which Section 10 has set to [red_action.task_id]). The CURRICULUM
        # is expected to already be detached by Section 10 — assert here so
        # we fail loudly if someone runs this cell out of context.
        cls = type(self)
        if self.RED_TASK_OVERRIDE:
            cls.STAGE_TASK_IDS = [self.RED_TASK_OVERRIDE]
        text = super().reset(**kwargs)
        self.red_metadata = dict(self.RED_METADATA)
        self.red_reward = 0.0
        print(
            f'[adv-env/reset] Red task override={self.RED_TASK_OVERRIDE} '
            f'metadata={self.red_metadata}'
        )
        return text

    def trajectory_payload(self) -> dict:
        base = super().trajectory_payload()
        last_step = self.trajectory[-1] if self.trajectory else {}
        downtime = float(last_step.get('downtime_minutes', 0.0))
        revenue = float(last_step.get('revenue_loss_usd', 0.0))
        unresolved = 0.0 if bool(base.get('is_resolved')) else 1.0
        steps = float(base.get('num_steps', 0))

        det_red_reward = (
            RED_REWARD_ALPHA_UNRESOLVED * unresolved
            + RED_REWARD_ALPHA_DOWNTIME * downtime
            + RED_REWARD_ALPHA_REVENUE * revenue
            + RED_REWARD_ALPHA_STEPS * steps
        )
        self.red_reward = float(det_red_reward)

        print(
            f'[adv-env/trajectory] resolved={bool(base.get("is_resolved"))} '
            f'downtime={downtime:.1f}m revenue=${revenue:.0f} steps={int(steps)} '
            f'det_red_r={det_red_reward:.3f}'
        )

        base['red_reward'] = self.red_reward
        base['det_red_reward'] = det_red_reward
        base['downtime_minutes'] = downtime
        base['revenue_loss_usd'] = revenue
        base['red_action'] = self.red_metadata
        return base


def validate_reward_structure_with_gemini(
    samples: List[dict],
    out_path: Path,
    stage: str = '',
) -> dict:
    """Run per-stage reward-logic sanity audit with Gemini.

    Checks:
    - Does higher blue success correlate with lower red reward?
    - Are reward bucket decompositions consistent with total_reward?
    - Are there suspicious reward spikes?

    Prints a clear PASS/WARN/FAIL verdict with issue list.
    """
    report = {
        'stage': stage,
        'checked_samples': len(samples),
        'gemini_used': False,
        'verdict': 'not_run',
        'issues': [],
        'suggested_adjustments': [],
    }

    # ── Deterministic pre-checks (always run) ──────────────────────────────
    det_issues = []
    for i, s in enumerate(samples):
        mismatch = abs(float(s.get('bucket_total_mismatch', 0.0)))
        if mismatch > 0.5:
            det_issues.append(
                f'Sample {i}: bucket_total_mismatch={mismatch:.3f} > 0.5 '
                f'(task={s.get("red_task_id","?")})'
            )
        blue_r = float(s.get('blue_total_reward', 0.0))
        red_r = float(s.get('red_reward', 0.0))
        resolved = bool(s.get('blue_resolved'))
        if resolved and red_r > RED_REWARD_ALPHA_UNRESOLVED * 0.8:
            det_issues.append(
                f'Sample {i}: Blue resolved but red_reward={red_r:.3f} seems high'
            )
        if abs(blue_r) > REWARD_AUDIT_MAX:
            det_issues.append(f'Sample {i}: blue_total_reward={blue_r:.3f} out of audit bounds')

    report['det_issues'] = det_issues
    if det_issues:
        print(f'[reward-audit/{stage}] Deterministic issues found:')
        for issue in det_issues:
            print(f'  ⚠ {issue}')
    else:
        print(f'[reward-audit/{stage}] Deterministic checks: all PASS')

    # ── Gemini audit ───────────────────────────────────────────────────────
    api_key = os.environ.get('GEMINI_API_KEY', '').strip()
    if not api_key:
        report['verdict'] = 'det_only_no_gemini'
        report['issues'] = det_issues
        out_path.write_text(json.dumps(report, indent=2))
        print(f'[reward-audit/{stage}] Gemini audit skipped (no API key) → det_only')
        return report

    print(f'[reward-audit/{stage}] Calling Gemini for reward structure audit...')

    prompt = json.dumps({
        'task': 'RL reward structure audit for SRE adversarial self-play',
        'stage': stage,
        'criteria': [
            'Does higher blue success rate correlate with lower red reward?',
            'Are total_reward and bucket decomposition (diag+fix+terminal) consistent?',
            'Are there reward spikes inconsistent with steps or outcome?',
            'Does det_red_reward behave sensibly (higher when Blue fails)?',
        ],
        'samples': samples[:8],
        'required_output': {
            'verdict': 'pass|warn|fail',
            'issues': ['list of specific issue strings'],
            'suggested_adjustments': ['list of concrete reward weight suggestions'],
        },
    })

    obj = _gemini_post(prompt, timeout=45)
    if obj:
        report.update(obj)
        report['gemini_used'] = True
        verdict = report.get('verdict', '?')
        issues = report.get('issues', [])
        adjustments = report.get('suggested_adjustments', [])

        verdict_emoji = {'pass': '✅', 'warn': '⚠️', 'fail': '❌'}.get(verdict, '?')
        print(f'[reward-audit/{stage}] Gemini verdict: {verdict_emoji} {verdict.upper()}')
        for issue in issues:
            print(f'  • {issue}')
        for adj in adjustments:
            print(f'  → {adj}')
    else:
        report['verdict'] = 'warn'
        report['issues'] = det_issues + ['gemini_audit_api_failed']
        print(f'[reward-audit/{stage}] Gemini audit FAILED → using det results only')

    out_path.write_text(json.dumps(report, indent=2))
    print(f'[reward-audit/{stage}] Report saved to {out_path}')
    return report


print('=' * 60)
print('Adversarial self-play utilities initialized.')
print(f'  USE_GEMINI_RED_AGENT : {USE_GEMINI_RED_AGENT}')
print(f'  GEMINI_MODEL         : {GEMINI_MODEL}')
print(f'  GEMINI_RED_WEIGHT    : {GEMINI_RED_WEIGHT}')
print(f'  Rounds/stage         : {SELF_PLAY_ROUNDS_PER_STAGE}')
print('=' * 60)


[adv-config] USE_GEMINI_RED_AGENT: False
[adv-config] GEMINI_MODEL: gemini-2.5-flash
[adv-config] GEMINI_RED_WEIGHT: 0.4
[adv-config] SELF_PLAY_ROUNDS_PER_STAGE: 2
Adversarial self-play utilities initialized.
  USE_GEMINI_RED_AGENT : False
  GEMINI_MODEL         : gemini-2.5-flash
  GEMINI_RED_WEIGHT    : 0.4
  Rounds/stage         : 2


## 10) Run Adversarial Self-Play Training + Reward Validation

This cell alternates Red and Blue each curriculum stage:

1. Red picks an adversarial task (Gemini if available, else heuristic).
2. Blue trains for one round against that task.
3. Episode trajectories are scored for both Blue and Red rewards.
4. Reward structure is validated and audited (including a Gemini audit when available).

In [18]:
import gc
import pandas as pd
from pathlib import Path
from datasets import Dataset
from trl import GRPOTrainer
import trackio

if not ENABLE_SELF_PLAY:
    print('ENABLE_SELF_PLAY=False, skipping adversarial run.')
else:
    adv_output_dir = output_dir / 'adversarial_self_play'
    adv_output_dir.mkdir(parents=True, exist_ok=True)

    # Detach the MasteryCurriculum + main RewardLogger for the duration of
    # adversarial training. Curriculum task picking would otherwise resample
    # within the stage pool, ignoring Red's choice; the main logger would
    # mix adversarial trajectories into the curriculum log.
    prev_curriculum = PageZeroToolEnv.CURRICULUM
    prev_logger = PageZeroToolEnv.REWARD_LOGGER
    PageZeroToolEnv.CURRICULUM = None
    AdversarialPageZeroToolEnv.CURRICULUM = None

    # Rebuild trainer using AdversarialPageZeroToolEnv. Same model / tokenizer
    # / LoRA config / single audited reward — only the environment_factory
    # changes so adversarial metadata flows through trajectory_payload.
    adv_trainer = GRPOTrainer(
        model=model,
        processing_class=tokenizer,
        reward_funcs=[reward_total_audited],
        train_dataset=make_stage_dataset(NUM_EPISODES, CURRICULUM_STAGES[0]['task_ids']),
        args=grpo_config,
        peft_config=peft_config,
        environment_factory=AdversarialPageZeroToolEnv,
    )

    red_agent = RedBreakerAgent()
    self_play_records = []
    stage_audit_reports = {}

    print(f'[self-play] Starting adversarial training: {len(CURRICULUM_STAGES)} stages')
    print(f'[self-play] Gemini Red reward: {"enabled" if USE_GEMINI_RED_AGENT else "disabled (heuristic only)"}')

    try:
        for stage in CURRICULUM_STAGES:
            stage_name = stage['name']
            stage_dir = adv_output_dir / f'stage_{stage_name}'
            stage_dir.mkdir(parents=True, exist_ok=True)

            # Per-stage RewardLogger so each stage gets its own
            # reward_log.csv + trajectories.jsonl pair (matches the
            # non-adversarial training layout under stage_<name>/).
            stage_logger = RewardLogger(
                stage_dir,
                csv_name=REWARD_LOG_NAME,
                jsonl_name=TRAJECTORY_LOG_NAME,
            )
            PageZeroToolEnv.REWARD_LOGGER = stage_logger
            AdversarialPageZeroToolEnv.REWARD_LOGGER = stage_logger

            stage_records_this = []  # records accumulated for this stage's audit

            for round_idx in range(SELF_PLAY_ROUNDS_PER_STAGE):
                # ── Red chooses adversarial task ──────────────────────────
                red_action = red_agent.choose(stage_name, stage['task_ids'])
                adv_metadata = {
                    'stage': stage_name,
                    'task_id': red_action.task_id,
                    'rationale': red_action.rationale,
                    'target_failure_mode': red_action.target_failure_mode,
                    'round': round_idx + 1,
                }
                # Pin the wrapper's stage pool to ONLY Red's chosen task —
                # CURRICULUM is None above, so STAGE_TASK_IDS wins.
                AdversarialPageZeroToolEnv._set_stage(stage_name, [red_action.task_id])
                AdversarialPageZeroToolEnv.set_red_attack(red_action.task_id, adv_metadata)

                round_dir = stage_dir / f'round_{round_idx + 1}'
                adv_trainer.train_dataset = make_stage_dataset(
                    stage['episodes'], [red_action.task_id]
                )
                adv_trainer.args.output_dir = str(round_dir)
                adv_trainer.args.run_name = (
                    f'pagezero-adv-{timestamp}-{stage_name}-r{round_idx + 1}'
                )
                if hasattr(adv_trainer.state, 'global_step'):
                    adv_trainer.state.global_step = 0
                adv_trainer.state.epoch = 0

                print()
                print('=' * 80)
                print(
                    f'[self-play] STAGE={stage_name.upper()} '
                    f'ROUND={round_idx + 1}/{SELF_PLAY_ROUNDS_PER_STAGE}'
                )
                print(f'  Red task      : {red_action.task_id}')
                print(f'  Red rationale : {red_action.rationale}')
                print(f'  Failure mode  : {red_action.target_failure_mode}')
                print(f'  Gemini Red    : {"enabled" if USE_GEMINI_RED_AGENT else "heuristic"}')
                print('=' * 80)

                # Re-initialize trackio for each training run to ensure proper logging
                trackio.init(project="openenv")

                # ── Blue trains against Red's chosen task ─────────────────
                try:
                    adv_trainer.train()
                finally:
                    AdversarialPageZeroToolEnv._close_all()

                # ── Collect this round's episodes from the stage JSONL ────
                round_episodes = []
                if stage_logger.jsonl_path.exists():
                    with open(stage_logger.jsonl_path) as f:
                        for line in f:
                            line = line.strip()
                            if not line:
                                continue
                            try:
                                row = json.loads(line)
                            except Exception:
                                continue
                            if row.get('stage') == stage_name:
                                round_episodes.append(row)
                tail = round_episodes[-stage['episodes']:] if round_episodes else []

                # ── Compute Red reward (det + Gemini blend) ───────────────
                for ep in tail:
                    det_r = float(ep.get('det_red_reward', ep.get('red_reward', 0.0)))
                    ep['det_red_reward'] = det_r

                det_red_mean = (
                    sum(float(ep.get('det_red_reward', 0.0)) for ep in tail)
                    / max(1, len(tail))
                )
                blended_red = red_agent.gemini_evaluate_red_reward(
                    red_action.task_id, tail, det_red_mean
                )
                print(
                    f'[self-play] Round red reward: '
                    f'det={det_red_mean:.3f} blended={blended_red:.3f}'
                )

                # ── Update Red agent stats ────────────────────────────────
                red_agent.update(red_action.task_id, tail, red_reward=blended_red)
                print(red_agent.debug_stats())

                # ── Accumulate records ────────────────────────────────────
                for ep in tail:
                    record = {
                        'stage': stage_name,
                        'round': round_idx + 1,
                        'red_task_id': red_action.task_id,
                        'red_rationale': red_action.rationale,
                        'red_failure_mode': red_action.target_failure_mode,
                        'blue_total_reward': ep.get('total_reward', 0.0),
                        'blue_diag_reward': ep.get('diagnosis_reward', 0.0),
                        'blue_fix_reward': ep.get('fix_reward', 0.0),
                        'blue_resolved': int(bool(ep.get('is_resolved'))),
                        'blue_steps': ep.get('num_steps', 0),
                        'downtime_minutes': ep.get('downtime_minutes', 0.0),
                        'revenue_loss_usd': ep.get('revenue_loss_usd', 0.0),
                        'det_red_reward': ep.get('det_red_reward', 0.0),
                        'blended_red_reward': blended_red,
                        'bucket_total_mismatch': ep.get('bucket_total_mismatch', 0.0),
                    }
                    self_play_records.append(record)
                    stage_records_this.append(record)

                # ── Save round checkpoint ─────────────────────────────────
                ckpt = round_dir / 'checkpoint'
                ckpt.mkdir(parents=True, exist_ok=True)
                adv_trainer.save_model(str(ckpt))
                print(f'[self-play] Round checkpoint saved: {ckpt}')

                gc.collect()
                try:
                    import torch
                    torch.cuda.empty_cache()
                except Exception:
                    pass

            # ── Per-stage reward audit (after all rounds in this stage) ───
            print(f'\n[self-play] Running per-stage reward audit for stage={stage_name}...')
            audit_path = stage_dir / 'gemini_reward_audit.json'
            audit = validate_reward_structure_with_gemini(
                stage_records_this, audit_path, stage=stage_name
            )
            stage_audit_reports[stage_name] = audit

            # Plot reward curve for this stage.
            try:
                plot_rewards(stage_logger.csv_path, stage_dir / 'reward_plot.png')
            except Exception as e:
                print(f'[self-play] Could not plot stage rewards: {e}')

    finally:
        # Always restore the main curriculum + logger so re-running
        # earlier cells (or kicking off another training run) sees the
        # original, unmodified state.
        PageZeroToolEnv.CURRICULUM = prev_curriculum
        PageZeroToolEnv.REWARD_LOGGER = prev_logger
        AdversarialPageZeroToolEnv.CURRICULUM = prev_curriculum
        AdversarialPageZeroToolEnv.REWARD_LOGGER = prev_logger

    # ── Final save and summary ────────────────────────────────────────────
    records_path = adv_output_dir / 'self_play_records.csv'
    pd.DataFrame(self_play_records).to_csv(records_path, index=False)
    print(f'\nSelf-play records saved to: {records_path}')

    # Global reward invariant check.
    if self_play_records:
        mismatches = [abs(float(r.get('bucket_total_mismatch', 0.0))) for r in self_play_records]
        max_mismatch = max(mismatches)
        print(f'[reward-check] max |bucket_total_mismatch| across all stages = {max_mismatch:.4f}')
        if max_mismatch > 1.0:
            print('[reward-check] WARNING: large decomposition mismatch — review llm_judge.py')

    print('\n=== Adversarial Self-Play Stage Audit Summary ===')
    for sname, audit in stage_audit_reports.items():
        verdict = audit.get('verdict', '?')
        gemini = '✅ Gemini' if audit.get('gemini_used') else '⚠ det-only'
        print(f'  {sname:6s} → {verdict.upper():4s} [{gemini}]')

    adv_final_dir = adv_output_dir / 'final'
    adv_final_dir.mkdir(parents=True, exist_ok=True)
    adv_trainer.save_model(str(adv_final_dir))
    print(f'\nAdversarial self-play model saved to: {adv_final_dir}')
    print(f'Red agent final stats: {red_agent.debug_stats()}')

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
/tmp/ipykernel_5512/125882343.py:26: UserWarning: You are using 'environment_factory', which is an experimental feature. This API may change or be removed at any time without prior notice. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  adv_trainer = GRPOTrainer(


[self-play] Starting adversarial training: 3 stages
[self-play] Gemini Red reward: disabled (heuristic only)
[red-agent/heuristic] Selected task_id=task_1 (score=0.900)

[self-play] STAGE=EASY ROUND=1/2
  Red task      : task_1
  Red rationale : heuristic: maximize unresolved+downtime likelihood
  Failure mode  : latency/cascading-failure
  Gemini Red    : heuristic
* Trackio project initialized: openenv
* Trackio metrics logged to: /root/.cache/huggingface/trackio
* Created new run: dainty-sunset-0


* Run finished. Uploading logs to Trackio (please wait...)
* Trackio project initialized: pagezero-rl
* Trackio metrics logged to: /root/.cache/huggingface/trackio
* Resumed existing run: pagezero-adv-2026-04-26_07-31-18-easy-r1


[nb-reset] stage=easy row_task=task_1 chosen=task_1
[adv-env/reset] Red task override=task_1 metadata={'stage': 'easy', 'task_id': 'task_1', 'rationale': 'heuristic: maximize unresolved+downtime likelihood', 'target_failure_mode': 'latency/cascading-failure', 'round': 1}
[nb-reset] stage=easy row_task=task_1 chosen=task_1
[adv-env/reset] Red task override=task_1 metadata={'stage': 'easy', 'task_id': 'task_1', 'rationale': 'heuristic: maximize unresolved+downtime likelihood', 'target_failure_mode': 'latency/cascading-failure', 'round': 1}
[nb-reset] stage=easy row_task=task_1 chosen=task_1
[adv-env/reset] Red task override=task_1 metadata={'stage': 'easy', 'task_id': 'task_1', 'rationale': 'heuristic: maximize unresolved+downtime likelihood', 'target_failure_mode': 'latency/cascading-failure', 'round': 1}
[nb-reset] stage=easy row_task=task_1 chosen=task_1
[adv-env/reset] Red task override=task_1 metadata={'stage': 'easy', 'task_id': 'task_1', 'rationale': 'heuristic: maximize unresolve

Step,Training Loss
1,1.498903
2,-0.500358
3,-0.499945
4,-0.502689
5,0.771915
6,-0.867311
7,-0.864436
8,0.946441
9,-0.497996
10,-0.498862


...

[nb-reset] stage=medium row_task=task_6 chosen=task_6
[adv-env/reset] Red task override=task_6 metadata={'stage': 'medium', 'task_id': 'task_6', 'rationale': 'heuristic: maximize unresolved+downtime likelihood', 'target_failure_mode': 'latency/cascading-failure', 'round': 2}
[nb-reset] stage=medium row_task=task_6 chosen=task_6
[adv-env/reset] Red task override=task_6 metadata={'stage': 'medium', 'task_id': 'task_6', 'rationale': 'heuristic: maximize unresolved+downtime likelihood', 'target_failure_mode': 'latency/cascading-failure', 'round': 2}
[nb-reset] stage=medium row_task=task_6 chosen=task_6
[adv-env/reset] Red task override=task_6 metadata={'stage': 'medium', 'task_id': 'task_6', 'rationale': 'heuristic: maximize unresolved+downtime likelihood', 'target_failure_mode': 'latency/cascading-failure', 'round': 2}
[nb-reset] stage=medium row_task=task_6 chosen=task_6
[adv-env/reset] Red task override=task_6 metadata={'stage': 'medium', 'task_id': 'task_6', 'rationale': 'heuristic: ma

Step,Training Loss
1,0.501366
2,0.496869
3,-1.504727
4,0.499924
5,0.505102
6,-1.506790
7,0.498440
8,0.496808
9,0.863976
10,0.864120


[nb-reset] stage=hard row_task=task_11 chosen=task_11
[adv-env/reset] Red task override=task_11 metadata={'stage': 'hard', 'task_id': 'task_11', 'rationale': 'heuristic: maximize unresolved+downtime likelihood', 'target_failure_mode': 'latency/cascading-failure', 'round': 1}
[nb-reset] stage=hard row_task=task_11 chosen=task_11
[adv-env/reset] Red task override=task_11 metadata={'stage': 'hard', 'task_id': 'task_11', 'rationale': 'heuristic: maximize unresolved+downtime likelihood', 'target_failure_mode': 'latency/cascading-failure', 'round': 1}
[nb-reset] stage=hard row_task=task_11 chosen=task_11
[adv-env/reset] Red task override=task_11 metadata={'stage': 'hard', 'task_id': 'task_11', 'rationale': 'heuristic: maximize unresolved+downtime likelihood', 'target_failure_mode': 'latency/cascading-failure', 'round': 1}
[nb-reset] stage=hard row_task=task_11 chosen=task_11
[adv-env/reset] Red task override=task_11 metadata={'stage': 'hard', 'task_id': 'task_11', 'rationale': 'heuristic: ma

Step,Training Loss
1,-0.728436
2,1.384041
3,0.067022
4,-0.726490
5,0.880240
6,-0.868466
7,-0.869402
8,0.861537
9,0.000038
10,0.000070


[nb-reset] stage=hard row_task=task_11 chosen=task_11
[adv-env/reset] Red task override=task_11 metadata={'stage': 'hard', 'task_id': 'task_11', 'rationale': 'heuristic: maximize unresolved+downtime likelihood', 'target_failure_mode': 'latency/cascading-failure', 'round': 2}
[nb-reset] stage=hard row_task=task_11 chosen=task_11
[adv-env/reset] Red task override=task_11 metadata={'stage': 'hard', 'task_id': 'task_11', 'rationale': 'heuristic: maximize unresolved+downtime likelihood', 'target_failure_mode': 'latency/cascading-failure', 'round': 2}
[nb-reset] stage=hard row_task=task_11 chosen=task_11
[adv-env/reset] Red task override=task_11 metadata={'stage': 'hard', 'task_id': 'task_11', 'rationale': 'heuristic: maximize unresolved+downtime likelihood', 'target_failure_mode': 'latency/cascading-failure', 'round': 2}
[nb-reset] stage=hard row_task=task_11 chosen=task_11
[adv-env/reset] Red task override=task_11 metadata={'stage': 'hard', 'task_id': 'task_11', 'rationale': 'heuristic: ma

## 11) Archive Outputs and View Server Log

Run after Sections 8–10 to bundle every artifact (baseline checkpoints, adversarial self-play outputs, reward logs, audits) into a single zip and to dump the most recent backend log.


In [20]:
import shutil

archive_name = output_dir.parent / f'{output_dir.name}.zip'
shutil.make_archive(str(archive_name.with_suffix('')), 'zip', output_dir)
print(f'Archived output folder to: {archive_name}')

Archived output folder to: outputs/pagezero-colab-2026-04-26_08-40-14.zip


In [21]:
import subprocess
import os

log_file_path = '/content/PageZero/server.log'
if os.path.exists(log_file_path):
    try:
        result = subprocess.run(['cat', log_file_path], capture_output=True, text=True, check=True)
        print(result.stdout)
    except subprocess.CalledProcessError as e:
        print(f"Error reading log file: {e}")
        print(f"Stderr: {e.stderr}")
    except FileNotFoundError:
        print(f"Log file not found at {log_file_path}")
else:
    print(f"Log file not found at {log_file_path}")

INFO:     Started server process [26559]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)



## 12) Latest Run Snapshot (2026-04-25)

**Run config**
- Model: `Qwen/Qwen3-1.7B`
- Episodes: `6`
- Generations: `6`
- Environment: `https://pranayy-pagezero-agent.hf.space/`
- Runtime: `36/36` steps in `46:43` (Epoch `1/1`)

**Training loss trace (steps 1-36)**

```text
1: 0.000000, 2: 0.228079, 3: -0.283213, 4: 0.005372, 5: 0.000000, 6: 0.000000,
7: 0.000000, 8: 0.000000, 9: 0.000000, 10: 0.062890, 11: 0.000000, 12: -0.297732,
13: 0.000000, 14: 0.000002, 15: 0.000001, 16: 0.000000, 17: 0.000000, 18: 0.000003,
19: 0.000000, 20: 0.000000, 21: 0.000004, 22: 0.000000, 23: 0.000000, 24: 0.000000,
25: 0.000000, 26: 0.000000, 27: 0.000000, 28: 0.000000, 29: 0.000000, 30: 0.000000,
31: 0.020289, 32: 0.281001, 33: -0.190548, 34: 0.000000, 35: -0.202171, 36: 0.000000
```

**Saved model**
- `outputs/pagezero-colab-2026-04-25_14-46-12`

**Reward curve summary (from plot)**
- Episodes: `36`
- Final average reward: `0.02`
- Best episode reward: `0.40`
- Trend: slightly positive, but near-flat and high-variance.

**Reward log tail**

| episode | total_reward | diagnosis_reward | fix_reward | timestamp |
|---:|---:|---:|---:|---|
| 27 | 0.0 | 0.0 | 0.0 | 2026-04-25T15:45:12.537719 |
| 28 | 0.0 | 0.0 | 0.0 | 2026-04-25T15:45:12.537743 |
| 29 | 0.0 | 0.0 | 0.0 | 2026-04-25T15:45:12.537767 |
| 30 | 0.0 | 0.0 | 0.0 | 2026-04-25T15:45:12.537790 |
| 31 | 0.2 | 0.0 | 0.2 | 2026-04-25T15:53:59.121829 |
| 32 | 0.0 | 0.0 | 0.0 | 2026-04-25T15:53:59.121913 |
| 33 | 0.0 | 0.0 | 0.0 | 2026-04-25T15:53:59.121943 |
| 34 | -0.2 | 0.0 | -0.2 | 2026-04-25T15:53:59.121967 |
| 35 | 0.2 | 0.2 | 0.0 | 2026-04-25T15:53:59.121991 |
| 36 | 0.0 | 0.0 | 0.0 | 2026-04-25T15:53:59.122014 |
